# Shortcut Features and Fair Evaluation in Phishing URL Detection
### Full experiment notebook

Reproduces every table and figure in the paper *"Shortcut Features and Fair Evaluation in Phishing URL Detection: A Statistical Comparison of Stacking Ensembles and Hybrid Feature Selection"*.

**How to run**
1. `Runtime → Run all` (CPU is enough; no GPU needed).
2. PhiUSIIL is downloaded automatically from the UCI repository. Place `dataset_phishing.csv` (WPD, see `data/README.md`) next to the notebook, or upload it when prompted.
3. The last cell writes `results/` (CSV and Excel tables, 300 dpi PNG/PDF figures) and `results.zip`.

**Runtime:** about 60 to 90 minutes on a free Colab CPU instance. `FAST_MODE = True` gives a quick trial on a subsample; reported results must come from the full run.

| Step | Paper section |
|---|---|
| 1 to 3 Data loading and description | 3 Datasets |
| 4 Preprocessing | 4.1 Preprocessing |
| 5 Shortcut analysis | 3.1 and 3.2 |
| 6 Hybrid feature selection | 4.2, 5.1 |
| 7 Models and tuning | 4.3, 4.4 |
| 8 to 10 Experiments 1 to 3 | 5.2, 5.3 |
| 11 Ablation | 5.3 |
| 12 Statistical tests | 4.6, 5.4 |
| 13 Figures | Figures |

## 0. Install dependencies

In [ ]:
!pip install -q ucimlrepo xgboost lightgbm openpyxl

## 1. Configuration
All experiment settings are defined here. `RANDOM_STATE` is fixed for reproducibility.

In [ ]:
import os, time, json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sklearn, xgboost, lightgbm
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_validate)
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif, RFECV
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                              VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, roc_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

# ---------------- experiment settings ----------------
RANDOM_STATE   = 42
TEST_SIZE      = 0.20     # 80/20 stratified split
CV_FOLDS       = 5        # folds used inside the Stacking model
FAST_MODE      = False    # True = quick trial on a subsample (NOT for the paper)
FAST_ROWS      = 30000

# Hybrid feature selection
MI_KEEP_RATIO  = 0.60     # stage 1: keep the top 60% of features by Mutual Information
MIN_FEATURES   = 5        # never go below this number of features
FS_SAMPLE      = 30000    # rows used for RFECV (speed); selection is applied to all rows

# Hyperparameter tuning (RandomizedSearchCV on a subsample)
TUNE           = True
TUNE_SAMPLE    = 30000
TUNE_ITER      = 15

# Statistical test (repeated CV: Stacking vs the best single model)
STAT_FOLDS     = 10
STAT_SAMPLE    = 40000

# Features to remove after the leakage check in step 5 (fill in if needed)
DROP_FEATURES_D1 = []
DROP_FEATURES_D2 = []

# PhiUSIIL feature setting
#   "url_only": only lexical features computed from the URL string itself. Realistic
#               scenario (decide before the page is opened) and avoids the shortcut
#               features that let every model reach ~100% on the full feature set.
#   "all":      every numeric feature (kept only for the shortcut demonstration).
D1_FEATURE_SET = "url_only"
URL_ONLY_FEATURES_D1 = [
    "URLLength", "DomainLength", "IsDomainIP", "CharContinuationRate",
    "TLDLegitimateProb", "URLCharProb", "TLDLength", "NoOfSubDomain",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio",
    "NoOfLettersInURL", "LetterRatioInURL", "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL", "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL", "IsHTTPS",
]   # URLSimilarityIndex is deliberately excluded (similarity to known legitimate URLs)

# Dataset 2 (Mendeley "Web page phishing detection", file: dataset_phishing.csv)
D2_FILE        = "dataset_phishing.csv"
D2_TARGET      = "status"
D2_POSITIVE    = "phishing"

OUT_DIR = "results"; FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
np.random.seed(RANDOM_STATE)

# ---------------- figure style (paper-ready) ----------------
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
           "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "legend.frameon": False,
    "lines.linewidth": 2,
})

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path + ".png"); plt.savefig(path + ".pdf")
    plt.show()

# Versions for the "Experimental Setup" paragraph of the paper
env = {"Python": platform.python_version(), "scikit-learn": sklearn.__version__,
       "XGBoost": xgboost.__version__, "LightGBM": lightgbm.__version__,
       "pandas": pd.__version__, "NumPy": np.__version__,
       "CPU cores": os.cpu_count(), "Platform": platform.platform()}
print(json.dumps(env, indent=2))

## 2. Load the data

**Dataset 1, PhiUSIIL** (Prasad and Chandra, *Computers & Security*, 2024): downloaded from UCI (id 967). The original label is `1 = legitimate`, `0 = phishing`; it is inverted so that **phishing = 1** (positive class).

**Dataset 2, WPD, Web page phishing detection** (Hannousse and Yahiouche, Mendeley Data, doi:10.17632/c2gw7fy2j4.3): use the file `dataset_B_05_2020.csv` from the Mendeley archive, renamed to `dataset_phishing.csv`.

In [ ]:
def load_d1():
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=967)
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        if ds.data.ids is not None:
            df = pd.concat([ds.data.ids, df], axis=1)
        print("PhiUSIIL downloaded from UCI.")
    except Exception as e:
        print("UCI download failed (", e, ") -> upload PhiUSIIL_Phishing_URL_Dataset.csv")
        from google.colab import files
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    return df

def load_d2():
    if not os.path.exists(D2_FILE):
        print(f"Upload {D2_FILE} (Mendeley / Kaggle: Web page phishing detection)")
        from google.colab import files
        up = files.upload()
        os.rename(next(iter(up)), D2_FILE)
    return pd.read_csv(D2_FILE)

df1_raw = load_d1()
df2_raw = load_d2()
print("D1:", df1_raw.shape, "| D2:", df2_raw.shape)

## 3. Dataset description
Produces the dataset summary table (Table 2 in the paper) and the class distribution figure.

In [ ]:
# make the target binary with phishing = 1
df1_raw["target"] = (df1_raw["label"] == 0).astype(int)
df1_raw = df1_raw.drop(columns=["label"])

assert D2_TARGET in df2_raw.columns, f"Column '{D2_TARGET}' not found. Columns: {list(df2_raw.columns)[:15]}..."
df2_raw["target"] = (df2_raw[D2_TARGET].astype(str).str.lower() == D2_POSITIVE).astype(int)
df2_raw = df2_raw.drop(columns=[D2_TARGET])

def describe(df, name):
    n = len(df); p = int(df["target"].sum())
    return {"Dataset": name, "Samples": n, "Raw columns": df.shape[1] - 1,
            "Phishing": p, "Legitimate": n - p,
            "Phishing %": round(100 * p / n, 2),
            "Missing values": int(df.isna().sum().sum()),
            "Duplicate rows": int(df.duplicated().sum())}

table_datasets = pd.DataFrame([describe(df1_raw, "PhiUSIIL"),
                               describe(df2_raw, "WPD")])
display(table_datasets)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, (df, name) in zip(axes, [(df1_raw, "PhiUSIIL"), (df2_raw, "WPD")]):
    counts = df["target"].value_counts().reindex([0, 1])
    bars = ax.bar(["Legitimate", "Phishing"], counts.values,
                  color=[PALETTE[0], PALETTE[1]], width=0.55)
    ax.bar_label(bars, labels=[f"{v:,}" for v in counts.values], padding=3, color=INK)
    ax.set_title(name); ax.set_ylabel("Number of URLs")
    ax.margins(y=0.15)
plt.tight_layout(); savefig("fig_class_distribution")

## 4. Preprocessing
Remove duplicate rows, drop raw text and identifier columns (URL, Domain, TLD, Title), encode low cardinality categorical columns, stratified 80/20 split, median imputation and removal of constant columns, then min and max scaling.

Medians, constant column checks and scaling bounds are fitted **on the training split only** and applied unchanged to the test split.

In [ ]:
def preprocess(df, name, drop_features=(), keep_only=None):
    log = {"Dataset": name, "Rows before": len(df)}
    df = df.drop_duplicates().reset_index(drop=True)
    log["Rows after dedup"] = len(df)
    y = df["target"].astype(int)
    X = df.drop(columns=["target"])

    obj_cols = list(X.select_dtypes(include=["object", "category", "bool"]).columns)
    low_card = [c for c in obj_cols if X[c].nunique() <= 20]
    high_card = [c for c in obj_cols if c not in low_card]          # raw text / identifiers
    X = X.drop(columns=high_card + [c for c in drop_features if c in X.columns])
    X = pd.get_dummies(X, columns=low_card, drop_first=True, dtype=float)
    X = X.apply(pd.to_numeric, errors="coerce").astype(float)
    log["Dropped text/ID columns"] = ", ".join(high_card) if high_card else "-"
    log["Manually dropped"] = ", ".join(drop_features) if drop_features else "-"
    if keep_only is not None:
        missing = [c for c in keep_only if c not in X.columns]
        if missing:
            print(f"{name}: these listed features were not found and are skipped -> {missing}")
        X = X[[c for c in keep_only if c in X.columns]]
        log["Feature set"] = f"restricted list ({X.shape[1]} features)"
    else:
        log["Feature set"] = "all numeric features"

    if FAST_MODE and len(X) > FAST_ROWS:
        X, _, y, _ = train_test_split(X, y, train_size=FAST_ROWS, stratify=y,
                                      random_state=RANDOM_STATE)

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y,
                                              random_state=RANDOM_STATE)
    med = X_tr.median()
    X_tr, X_te = X_tr.fillna(med), X_te.fillna(med)
    const = list(X_tr.columns[X_tr.nunique() <= 1])
    X_tr, X_te = X_tr.drop(columns=const), X_te.drop(columns=const)
    scaler = MinMaxScaler().fit(X_tr)
    X_tr = pd.DataFrame(scaler.transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te = pd.DataFrame(scaler.transform(X_te), columns=X_te.columns, index=X_te.index)

    log.update({"Constant columns removed": len(const), "Final features": X_tr.shape[1],
                "Train size": len(X_tr), "Test size": len(X_te)})
    return X_tr, X_te, y_tr.reset_index(drop=True), y_te.reset_index(drop=True), log

# full PhiUSIIL feature set (used only to demonstrate the shortcut problem in step 5)
X1f_tr, X1f_te, y1f_tr, y1f_te, log1f = preprocess(df1_raw, "PhiUSIIL (all features)", DROP_FEATURES_D1)
# PhiUSIIL as used in the main experiments
X1_tr, X1_te, y1_tr, y1_te, log1 = preprocess(
    df1_raw, "PhiUSIIL", DROP_FEATURES_D1,
    keep_only=URL_ONLY_FEATURES_D1 if D1_FEATURE_SET == "url_only" else None)
X1f_tr, X1f_te = X1f_tr.reset_index(drop=True), X1f_te.reset_index(drop=True)
X2_tr, X2_te, y2_tr, y2_te, log2 = preprocess(df2_raw, "WPD", DROP_FEATURES_D2)
X1_tr, X1_te = X1_tr.reset_index(drop=True), X1_te.reset_index(drop=True)
X2_tr, X2_te = X2_tr.reset_index(drop=True), X2_te.reset_index(drop=True)
table_preprocessing = pd.DataFrame([log1f, log1, log2])
display(table_preprocessing.T)

## 5. Shortcut analysis
The first cell computes the ROC-AUC of every feature on its own (training split). Features above 0.95 are shortcut candidates; the cell warns at 0.99.

The main experiments use `D1_FEATURE_SET = "url_only"` (the lexical setting in the paper): 21 features computed from the URL string, excluding URLSimilarityIndex and all page content counts.

In [ ]:
def single_feature_auc(X, y):
    out = {}
    for c in X.columns:
        try:
            a = roc_auc_score(y, X[c])
            out[c] = max(a, 1 - a)
        except ValueError:
            pass
    return pd.Series(out).sort_values(ascending=False)

leak_rows = []
for name, X, y in [("PhiUSIIL (all features)", X1f_tr, y1f_tr), ("PhiUSIIL", X1_tr, y1_tr),
                   ("WPD", X2_tr, y2_tr)]:
    auc = single_feature_auc(X, y)
    leak_rows += [{"Dataset": name, "Rank": i + 1, "Feature": f, "Single-feature AUC": round(v, 4)}
                  for i, (f, v) in enumerate(auc.head(10).items())]
    sus = auc[auc >= 0.99]
    print(f"\n{name} - top 8 single-feature AUCs:")
    display(auc.head(8).round(4).to_frame("AUC"))
    if len(sus):
        print(f"WARNING ({name}): possible shortcut features ->", list(sus.index))
    else:
        print(f"{name}: no single feature reaches AUC 0.99. OK.")
table_leakage = pd.DataFrame(leak_rows)

### 5b. Saturation of the full PhiUSIIL feature set
Three reference models on the full feature set and on the lexical setting (Table 4 in the paper).

In [ ]:
quick = {"Logistic Regression": lambda: LogisticRegression(max_iter=2000),
         "Decision Tree": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
         "LightGBM": lambda: LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE, verbose=-1)}
sc_rows = []
for setting, (Xa, Xb, ya, yb) in {"All features": (X1f_tr, X1f_te, y1f_tr, y1f_te),
                                  "URL-only features": (X1_tr, X1_te, y1_tr, y1_te)}.items():
    for m, f in quick.items():
        mdl = f().fit(Xa, ya); pr = mdl.predict_proba(Xb)[:, 1]; pd_ = (pr >= 0.5).astype(int)
        sc_rows.append({"Setting": setting, "Features": Xa.shape[1], "Model": m,
                        "Accuracy": accuracy_score(yb, pd_), "F1": f1_score(yb, pd_),
                        "ROC-AUC": roc_auc_score(yb, pr)})
table_shortcut = pd.DataFrame(sc_rows)
show = table_shortcut.copy()
for c in ["Accuracy", "F1", "ROC-AUC"]: show[c] = (show[c] * 100).round(2)
display(show)

## 6. Hybrid feature selection
**Stage 1, Mutual Information filter:** features are ranked by

$$I(X;Y)=\sum_{x}\sum_{y} p(x,y)\log\frac{p(x,y)}{p(x)\,p(y)}$$

and the top `MI_KEEP_RATIO` (60%) are kept.

**Stage 2, RFECV wrapper:** recursive feature elimination with a Random Forest (100 trees), 5-fold stratified CV and F1 scoring.

Selection uses the training split only.

In [ ]:
def subsample(X, y, n):
    if len(X) <= n:
        return X, y
    Xs, _, ys, _ = train_test_split(X, y, train_size=n, stratify=y, random_state=RANDOM_STATE)
    return Xs, ys

def rf_for_rfe():
    return RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)

def run_rfecv(X, y, features):
    rfe = RFECV(rf_for_rfe(), step=1, scoring="f1", n_jobs=-1,
                min_features_to_select=min(MIN_FEATURES, len(features)),
                cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
    rfe.fit(X[features], y)
    return [f for f, keep in zip(features, rfe.support_) if keep], rfe

def hybrid_feature_selection(X_tr, y_tr, name):
    t0 = time.perf_counter()
    Xs, ys = subsample(X_tr, y_tr, FS_SAMPLE)
    mi = pd.Series(mutual_info_classif(Xs, ys, random_state=RANDOM_STATE),
                   index=X_tr.columns).sort_values(ascending=False)
    k = max(MIN_FEATURES, int(np.ceil(len(mi) * MI_KEEP_RATIO)))
    stage1 = list(mi.index[:k])
    selected, rfe = run_rfecv(Xs, ys, stage1)
    fs_time = time.perf_counter() - t0
    print(f"{name}: {X_tr.shape[1]} -> {len(stage1)} (MI) -> {len(selected)} (RFECV)"
          f"  [{fs_time:.1f}s]")
    return {"mi": mi, "stage1": stage1, "selected": selected, "rfe": rfe, "time": fs_time}

fs1 = hybrid_feature_selection(X1_tr, y1_tr, "PhiUSIIL")
fs2 = hybrid_feature_selection(X2_tr, y2_tr, "WPD")

# RFE-only (for the ablation study): RFECV on all features, no MI stage
Xs, ys = subsample(X1_tr, y1_tr, FS_SAMPLE)
rfe_only_1, _ = run_rfecv(Xs, ys, list(X1_tr.columns))
print("PhiUSIIL RFE-only:", len(rfe_only_1), "features")

table_fs = pd.DataFrame([
    {"Dataset": n, "Original": X.shape[1], "After MI": len(f["stage1"]),
     "After RFECV (final)": len(f["selected"]),
     "Reduction %": round(100 * (1 - len(f["selected"]) / X.shape[1]), 1),
     "Selection time (s)": round(f["time"], 1),
     "Selected features": ", ".join(f["selected"])}
    for n, X, f in [("PhiUSIIL", X1_tr, fs1), ("WPD", X2_tr, fs2)]])
display(table_fs)

## 7. Models and hyperparameter tuning
- Baselines: logistic regression, decision tree, Gaussian naive Bayes.
- Base learners: Random Forest, XGBoost, LightGBM.
- Stacking: the three base learners with a logistic regression meta-learner trained on 5-fold out-of-fold probabilities.

Tuning: `RandomizedSearchCV` (15 candidates, 3-fold CV, F1) on a training sample. The selected values form Table 5 in the paper.

In [ ]:
DEFAULT_PARAMS = {
    "RF": dict(n_estimators=200, max_depth=None, min_samples_split=2, max_features="sqrt"),
    "XGBoost": dict(n_estimators=300, learning_rate=0.1, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8),
    "LightGBM": dict(n_estimators=300, learning_rate=0.1, num_leaves=63,
                     subsample=0.8, subsample_freq=1, colsample_bytree=0.8),
}
SEARCH_SPACE = {
    "RF": {"n_estimators": [100, 200, 300], "max_depth": [None, 20, 30],
           "min_samples_split": [2, 5, 10], "max_features": ["sqrt", "log2"]},
    "XGBoost": {"n_estimators": [200, 300, 500], "learning_rate": [0.03, 0.05, 0.1, 0.2],
                "max_depth": [4, 6, 8, 10], "subsample": [0.7, 0.8, 1.0],
                "colsample_bytree": [0.7, 0.8, 1.0]},
    "LightGBM": {"n_estimators": [200, 300, 500], "learning_rate": [0.03, 0.05, 0.1, 0.2],
                 "num_leaves": [31, 63, 127], "subsample": [0.7, 0.8, 1.0],
                 "colsample_bytree": [0.7, 0.8, 1.0]},
}

def make_base(name, params):
    p = dict(params)
    if name == "RF":
        return RandomForestClassifier(**p, n_jobs=-1, random_state=RANDOM_STATE)
    if name == "XGBoost":
        return XGBClassifier(**p, eval_metric="logloss", tree_method="hist",
                             n_jobs=-1, random_state=RANDOM_STATE)
    if name == "LightGBM":
        p.setdefault("subsample_freq", 1)
        return LGBMClassifier(**p, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)

def tune(X_tr, y_tr, features, name):
    params = {k: dict(v) for k, v in DEFAULT_PARAMS.items()}
    rows = []
    if not TUNE:
        return params, pd.DataFrame()
    Xs, ys = subsample(X_tr[features], y_tr, TUNE_SAMPLE)
    for m in params:
        search = RandomizedSearchCV(make_base(m, params[m]), SEARCH_SPACE[m],
                                    n_iter=TUNE_ITER, scoring="f1", n_jobs=1,
                                    cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                    random_state=RANDOM_STATE)
        search.fit(Xs, ys)
        params[m].update(search.best_params_)
        rows.append({"Dataset": name, "Model": m, "Best CV F1": round(search.best_score_, 4),
                     **{k: v for k, v in search.best_params_.items()}})
        print(f"{name} | {m}: {search.best_params_}  (F1={search.best_score_:.4f})")
    return params, pd.DataFrame(rows)

params1, table_hp1 = tune(X1_tr, y1_tr, fs1["selected"], "PhiUSIIL")
params2, table_hp2 = tune(X2_tr, y2_tr, fs2["selected"], "WPD")
table_hyperparams = pd.concat([table_hp1, table_hp2], ignore_index=True)
display(table_hyperparams)

def all_models(params, include=("RF", "XGBoost", "LightGBM")):
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Naive Bayes": GaussianNB(),
        **{m: make_base(m, params[m]) for m in ("RF", "XGBoost", "LightGBM")},
        "Proposed Stacking": make_stacking(params, include),
    }

def make_stacking(params, include=("RF", "XGBoost", "LightGBM")):
    return StackingClassifier(
        estimators=[(m, make_base(m, params[m])) for m in include],
        final_estimator=LogisticRegression(max_iter=2000),
        stack_method="predict_proba", passthrough=False, n_jobs=1,
        cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=RANDOM_STATE))

def make_voting(params, include=("RF", "XGBoost", "LightGBM")):
    return VotingClassifier([(m, make_base(m, params[m])) for m in include], voting="soft")

### Evaluation helper
Metrics: accuracy, precision, recall, F1, ROC-AUC, MCC, FPR, training time and prediction time per 1,000 URLs.

In [ ]:
PRED_STORE = {}   # (dataset, setting, model) -> (y_true, proba)

def evaluate(model, X_tr, y_tr, X_te, y_te, model_name, dataset, setting):
    t0 = time.perf_counter(); model.fit(X_tr, y_tr); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); proba = model.predict_proba(X_te)[:, 1]
    t_pred = time.perf_counter() - t0
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    PRED_STORE[(dataset, setting, model_name)] = (np.asarray(y_te), proba)
    row = {"Dataset": dataset, "Setting": setting, "Model": model_name,
           "Features": X_tr.shape[1],
           "Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
           "Recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred),
           "ROC-AUC": roc_auc_score(y_te, proba), "MCC": matthews_corrcoef(y_te, pred),
           "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
           "Train time (s)": t_train, "Predict ms / 1k URLs": 1000 * t_pred / len(X_te) * 1000}
    print(f"  {model_name:<22} F1={row['F1']:.4f}  Acc={row['Accuracy']:.4f}  "
          f"AUC={row['ROC-AUC']:.4f}  train={t_train:.1f}s")
    return row

METRIC_COLS = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "MCC", "FPR"]
def fmt(df):
    out = df.copy()
    for c in METRIC_COLS:
        if c in out: out[c] = (out[c] * 100).round(2)      # percentages, 2 decimals
    for c in ["Train time (s)", "Predict ms / 1k URLs"]:
        if c in out: out[c] = out[c].round(3)
    return out

## 8. Experiment 1: all models on PhiUSIIL (lexical setting, selected features)
Table 6 in the paper.

In [ ]:
sel1 = fs1["selected"]
rows = []
for name, model in all_models(params1).items():
    rows.append(evaluate(model, X1_tr[sel1], y1_tr, X1_te[sel1], y1_te,
                         name, "PhiUSIIL", "HFS"))
table_exp1 = pd.DataFrame(rows)
display(fmt(table_exp1).drop(columns=["Dataset", "Setting"]))

## 9. Experiment 2: effect of feature selection
The strongest models on all 21 lexical features against the selected subset (Table 8).

In [ ]:
rows = []
for name in ["RF", "XGBoost", "LightGBM", "Proposed Stacking"]:
    model = all_models(params1)[name]
    rows.append(evaluate(model, X1_tr, y1_tr, X1_te, y1_te, name, "PhiUSIIL", "All features"))
table_all_feat = pd.DataFrame(rows)
table_exp2 = pd.concat([table_all_feat,
                        table_exp1[table_exp1["Model"].isin(table_all_feat["Model"])]],
                       ignore_index=True).sort_values(["Model", "Setting"])
display(fmt(table_exp2)[["Model", "Setting", "Features", "Accuracy", "F1", "ROC-AUC",
                         "Train time (s)", "Predict ms / 1k URLs"]])

## 10. Experiment 3: second dataset (WPD)
The same pipeline on WPD (Table 7).

In [ ]:
sel2 = fs2["selected"]
rows = []
for name, model in all_models(params2).items():
    rows.append(evaluate(model, X2_tr[sel2], y2_tr, X2_te[sel2], y2_te,
                         name, "WPD", "HFS"))
table_exp3 = pd.DataFrame(rows)
display(fmt(table_exp3).drop(columns=["Dataset", "Setting"]))

## 11. Ablation study
1. Full framework (MI + RFECV + stacking)
2. Without feature selection
3. MI filter only
4. RFECV only
5 to 7. Stacking without RF, without XGBoost, without LightGBM
8. Soft voting instead of the meta-learner

In [ ]:
full = table_exp1[table_exp1["Model"] == "Proposed Stacking"].iloc[0].to_dict()
nofs = table_all_feat[table_all_feat["Model"] == "Proposed Stacking"].iloc[0].to_dict()
ab_rows = [dict(full, Variant="Full framework (MI + RFECV + Stacking)"),
           dict(nofs, Variant="w/o feature selection")]

def ab(variant, model, feats):
    r = evaluate(model, X1_tr[feats], y1_tr, X1_te[feats], y1_te, variant, "PhiUSIIL", "Ablation")
    r["Variant"] = variant; ab_rows.append(r)

ab("MI only (no RFECV)",     make_stacking(params1), fs1["stage1"])
ab("RFECV only (no MI)",     make_stacking(params1), rfe_only_1)
ab("Stacking w/o RF",        make_stacking(params1, ("XGBoost", "LightGBM")), sel1)
ab("Stacking w/o XGBoost",   make_stacking(params1, ("RF", "LightGBM")), sel1)
ab("Stacking w/o LightGBM",  make_stacking(params1, ("RF", "XGBoost")), sel1)
ab("Soft voting instead of meta-learner", make_voting(params1), sel1)

table_ablation = pd.DataFrame(ab_rows)
table_ablation["ΔF1 vs full (pp)"] = (table_ablation["F1"] - full["F1"]) * 100
display(fmt(table_ablation)[["Variant", "Features", "Accuracy", "F1", "ROC-AUC", "MCC",
                             "ΔF1 vs full (pp)", "Train time (s)"]].round(3))

## 12. Statistical tests
10-fold stratified CV on identical folds for stacking and each base learner, on both datasets. Paired F1 scores are compared with:
- the corrected resampled t-test of Nadeau and Bengio (2003), which accounts for overlapping training folds;
- the uncorrected paired t-test (reported for reference);
- the two-sided Wilcoxon signed-rank test.

P values of the corrected t-test and the Wilcoxon test are adjusted with the Holm procedure over the six comparisons. Per-fold scores are saved to `results/cv_fold_scores.csv`.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals, float); n = len(p); order = np.argsort(p)
    adj = np.empty(n); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (n - rank) * p[i])); adj[i] = running
    return adj

def corrected_ttest(diff, n_train, n_test):
    """Nadeau and Bengio (2003) corrected resampled t-test."""
    k = len(diff); m = diff.mean(); v = diff.var(ddof=1)
    if v == 0:
        return 1.0
    t = m / np.sqrt((1.0 / k + n_test / n_train) * v)
    return 2 * stats.t.sf(abs(t), k - 1)

FOLD_ROWS = []
def stat_compare(X_tr, y_tr, feats, params, dataset, n_rows):
    Xs, ys = subsample(X_tr[feats], y_tr, n_rows)
    cv = StratifiedKFold(STAT_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    n_test = len(Xs) / STAT_FOLDS; n_train = len(Xs) - n_test
    f1_stack = cross_validate(make_stacking(params), Xs, ys, cv=cv, scoring="f1")["test_score"]
    rows = []
    for m in ["RF", "XGBoost", "LightGBM"]:
        f1_m = cross_validate(make_base(m, params[m]), Xs, ys, cv=cv, scoring="f1")["test_score"]
        diff = f1_stack - f1_m
        for k_, (a, b_) in enumerate(zip(f1_stack, f1_m)):
            FOLD_ROWS.append({"Dataset": dataset, "Comparison": f"Stacking vs {m}", "Fold": k_ + 1,
                              "Stacking F1": a, "Other F1": b_})
        t_p = stats.ttest_rel(f1_stack, f1_m).pvalue if np.any(diff) else 1.0
        tc_p = corrected_ttest(diff, n_train, n_test) if np.any(diff) else 1.0
        w_p = stats.wilcoxon(f1_stack, f1_m).pvalue if np.any(diff) else 1.0
        rows.append({"Dataset": dataset, "Comparison": f"Stacking vs {m}",
                     "Folds": STAT_FOLDS, "Rows used": len(Xs),
                     "Stacking F1 mean±std (%)": f"{100*f1_stack.mean():.2f} ± {100*f1_stack.std():.2f}",
                     "Other F1 mean±std (%)": f"{100*f1_m.mean():.2f} ± {100*f1_m.std():.2f}",
                     "Mean diff (pp)": round(100 * diff.mean(), 3),
                     "Paired t-test p (uncorrected)": t_p,
                     "Corrected t-test p": tc_p, "Wilcoxon p": w_p})
        print(dataset, "|", f"Stacking vs {m}", "| corrected t p =", round(tc_p, 4), "| Wilcoxon p =", round(w_p, 4))
    return rows

table_stats = pd.DataFrame(
    stat_compare(X1_tr, y1_tr, sel1, params1, "PhiUSIIL", STAT_SAMPLE) +
    stat_compare(X2_tr, y2_tr, sel2, params2, "WPD", len(X2_tr)))
table_stats["Corrected t-test p (Holm)"] = holm(table_stats["Corrected t-test p"])
table_stats["Wilcoxon p (Holm)"] = holm(table_stats["Wilcoxon p"])
def outcome(r):
    sig_t = r["Corrected t-test p (Holm)"] < 0.05; sig_w = r["Wilcoxon p (Holm)"] < 0.05
    if not (sig_t or sig_w): return "Not significant"
    side = "Stacking better" if r["Mean diff (pp)"] > 0 else "Stacking worse"
    return side if (sig_t and sig_w) else side + (" (Wilcoxon only)" if sig_w else " (t-test only)")
table_stats["Outcome"] = table_stats.apply(outcome, axis=1)
for c in ["Paired t-test p (uncorrected)", "Corrected t-test p", "Wilcoxon p",
          "Corrected t-test p (Holm)", "Wilcoxon p (Holm)"]:
    table_stats[c] = table_stats[c].round(5)
table_folds = pd.DataFrame(FOLD_ROWS)
display(table_stats)

## 13. Figures
All figures are saved as PNG and PDF (300 dpi) in `results/figures`.

In [ ]:
# Fig: top Mutual Information scores
top = fs1["mi"].head(20)[::-1]
colors = [PALETTE[0] if f in sel1 else "#b9b8b2" for f in top.index]
plt.figure(figsize=(7, 5.5))
plt.barh(top.index, top.values, color=colors, height=0.65)
plt.xlabel("Mutual Information score"); plt.title("Top-20 features by Mutual Information (PhiUSIIL)")
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=PALETTE[0], label="Selected by RFECV"),
                    Patch(color="#b9b8b2", label="Not selected")], loc="lower right")
plt.grid(axis="y", visible=False)
savefig("fig_mi_scores")

# Fig: RFECV curve (number of features vs CV F1)
for fs, name in [(fs1, "PhiUSIIL"), (fs2, "WPD")]:
    res = fs["rfe"].cv_results_
    n_feat = res.get("n_features", np.arange(fs["rfe"].min_features_to_select,
                                            fs["rfe"].min_features_to_select + len(res["mean_test_score"])))
    mean, std = res["mean_test_score"], res["std_test_score"]
    plt.figure(figsize=(6, 3.6))
    plt.plot(n_feat, mean, color=PALETTE[0], marker="o", markersize=4, label="Mean CV F1")
    plt.fill_between(n_feat, mean - std, mean + std, color=PALETTE[0], alpha=0.15, label="± 1 std")
    plt.axvline(len(fs["selected"]), color=PALETTE[1], linestyle="--", linewidth=1.5,
                label=f"Selected: {len(fs['selected'])} features")
    plt.xlabel("Number of features"); plt.ylabel("F1 (5-fold CV)")
    plt.title(f"RFECV feature elimination ({name})"); plt.legend(loc="lower right")
    savefig(f"fig_rfecv_{name.lower().replace('-', '_')}")

# Fig: model comparison (grouped bars) for both datasets
for tab, name in [(table_exp1, "PhiUSIIL"), (table_exp3, "WPD")]:
    metrics = ["Accuracy", "Precision", "Recall", "F1"]
    d = tab.set_index("Model")[metrics] * 100
    x = np.arange(len(d)); w = 0.2
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for i, m in enumerate(metrics):
        ax.bar(x + (i - 1.5) * w, d[m], width=w - 0.02, color=PALETTE[i], label=m)
    ax.set_xticks(x); ax.set_xticklabels(d.index, rotation=15, ha="right")
    lo = max(0, np.floor(d.values.min() / 5) * 5 - 5)
    ax.set_ylim(lo, 100.5); ax.set_ylabel("Score (%)")
    ax.set_title(f"Performance comparison of all models ({name})")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.28))
    ax.grid(axis="x", visible=False)
    savefig(f"fig_model_comparison_{name.lower().replace('-', '_')}")

# Fig: confusion matrices of the proposed model
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, ds in zip(axes, ["PhiUSIIL", "WPD"]):
    yt, pr = PRED_STORE[(ds, "HFS", "Proposed Stacking")]
    cm = confusion_matrix(yt, (pr >= 0.5).astype(int))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Legitimate", "Phishing"], yticklabels=["Legitimate", "Phishing"],
                annot_kws={"size": 11})
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(ds)
plt.suptitle("Confusion matrices of the proposed Stacking model", fontweight="bold")
plt.tight_layout(); savefig("fig_confusion_matrices")

# Fig: ROC curves
for ds in ["PhiUSIIL", "WPD"]:
    plt.figure(figsize=(5.2, 4.6))
    models = [k[2] for k in PRED_STORE if k[0] == ds and k[1] == "HFS"]
    for i, m in enumerate(models):
        yt, pr = PRED_STORE[(ds, "HFS", m)]
        fpr, tpr, _ = roc_curve(yt, pr)
        lw = 2.6 if m == "Proposed Stacking" else 1.6
        plt.plot(fpr, tpr, color=PALETTE[i % len(PALETTE)], linewidth=lw,
                 label=f"{m} (AUC={roc_auc_score(yt, pr):.4f})")
    plt.plot([0, 1], [0, 1], color="#b9b8b2", linestyle=":", linewidth=1.2)
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title(f"ROC curves ({ds})"); plt.legend(loc="lower right", fontsize=8)
    savefig(f"fig_roc_{ds.lower().replace('-', '_')}")

# Fig: effect of feature selection (F1 and training time)
d = table_exp2.pivot(index="Model", columns="Setting", values=["F1", "Train time (s)"])
models = list(d.index); x = np.arange(len(models)); w = 0.38
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, metric, scale, unit in [(axes[0], "F1", 100, "F1 (%)"),
                                (axes[1], "Train time (s)", 1, "Training time (s)")]:
    for i, (setting, c) in enumerate([("All features", "#b9b8b2"), ("HFS", PALETTE[0])]):
        vals = d[(metric, setting)] * scale
        bars = ax.bar(x + (i - 0.5) * w, vals, width=w - 0.03, color=c,
                      label=f"{setting} ({table_exp2[table_exp2.Setting == setting].Features.iloc[0]} features)")
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=2, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=10); ax.set_ylabel(unit)
    ax.grid(axis="x", visible=False)
    if metric == "F1":
        v = d["F1"].values * 100; ax.set_ylim(max(0, v.min() - 2), min(100.6, v.max() + 1))
axes[0].set_title("Detection performance"); axes[1].set_title("Training cost")
h, l = axes[1].get_legend_handles_labels()
fig.legend(h, l, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.08))
plt.suptitle("Effect of hybrid feature selection (PhiUSIIL)", fontweight="bold")
plt.tight_layout(rect=(0, 0.04, 1, 1)); savefig("fig_feature_selection_effect")

## 14. Save all results
Writes `results/paper_tables.xlsx` (one sheet per table), a CSV per table, the figures, `environment.json`, and `results.zip`.

In [ ]:
tables = {
    "1_datasets": table_datasets, "2_preprocessing": table_preprocessing,
    "3_feature_selection": table_fs, "4_hyperparameters": table_hyperparams,
    "5_exp1_models_D1": fmt(table_exp1), "6_exp2_fs_effect": fmt(table_exp2),
    "7_exp3_models_D2": fmt(table_exp3), "8_ablation": fmt(table_ablation),
    "9_statistical_test": table_stats, "10_leakage_auc": table_leakage,
    "11_shortcut_demo": table_shortcut, "12_cv_fold_scores": table_folds,
}
with pd.ExcelWriter(os.path.join(OUT_DIR, "paper_tables.xlsx")) as xw:
    for name, t in tables.items():
        t.to_excel(xw, sheet_name=name[:31], index=False)
        t.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)
with open(os.path.join(OUT_DIR, "environment.json"), "w") as f:
    json.dump({**env, "FAST_MODE": FAST_MODE, "RANDOM_STATE": RANDOM_STATE}, f, indent=2)

import shutil
shutil.make_archive("results", "zip", OUT_DIR)
print("Saved:", sorted(os.listdir(OUT_DIR)))
if FAST_MODE:
    print("\nNOTE: FAST_MODE=True -> these numbers come from a subsample trial run.")
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    pass